In [2]:
from __future__ import annotations

import json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import types

import torch
from transformers import AutoModel, AutoTokenizer

# ===================== CONFIG =====================
CACHE_JSON = Path(r"metaspace_images_dump\name_to_pubchem_struct_cache.json")
OUT_NPY    = Path(r"metaspace_images_dump\molformer_pubchem_embeddings.npy")
OUT_INDEX  = Path(r"metaspace_images_dump\molformer_pubchem_index.parquet")

MODEL_NAME = "ibm/MoLFormer-XL-both-10pct"
BATCH_SIZE = 64
MAX_LEN    = 256   # adjust if you hit truncation issues
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# ===================== LOAD MODEL =====================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    deterministic_eval=True,
    trust_remote_code=True,
).to(DEVICE).eval()

if not hasattr(model, "warn_if_padding_and_no_attention_mask"):
    def _noop_warn(self, input_ids=None, attention_mask=None, **kwargs):
        return
    model.warn_if_padding_and_no_attention_mask = types.MethodType(_noop_warn, model)

@torch.no_grad()
def embed_smiles_list(smiles_list: list[str]) -> np.ndarray:
    """Returns [N, D] float32 embeddings using pooler_output."""
    all_emb = []
    for i in tqdm(range(0, len(smiles_list), BATCH_SIZE), desc="Embedding (MoLFormer)"):
        batch = smiles_list[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        ).to(DEVICE)

        out = model(**inputs)
        emb = out.pooler_output  # [B, D]
        all_emb.append(emb.detach().cpu().numpy().astype(np.float32))
    return np.vstack(all_emb) if all_emb else np.zeros((0, 0), dtype=np.float32)

def main():
    cache = json.loads(CACHE_JSON.read_text(encoding="utf-8"))

    # Build a table of unique (cid_used, smiles). If duplicate CID appears with same smiles, keep one.
    rows = []
    for name_key, rec in cache.items():
        if not isinstance(rec, dict):
            continue
        if not rec.get("ok"):
            continue
        cid = rec.get("cid_used")
        smi = rec.get("smiles")
        if not cid or not smi:
            continue
        rows.append(
            {
                "name": name_key,
                "cid": f"CID{int(cid)}",
                "smiles": str(smi),
                "inchi": rec.get("inchikey"),
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No ok SMILES found in cache JSON.")

    # Deduplicate by CID (or by SMILES if you prefer)
    df = df.drop_duplicates(subset=["cid"]).reset_index(drop=True)

    smiles = df["smiles"].tolist()
    emb = embed_smiles_list(smiles)

    # Save
    OUT_NPY.parent.mkdir(parents=True, exist_ok=True)
    np.save(OUT_NPY, emb)
    df.to_parquet(OUT_INDEX, index=False)

    print(f"[DONE] embeddings: {OUT_NPY}  shape={emb.shape}")
    print(f"[DONE] index:      {OUT_INDEX}  rows={len(df):,}")

if __name__ == "__main__":
    main()


Embedding (MoLFormer): 100%|██████████| 477/477 [01:25<00:00,  5.58it/s]


[DONE] embeddings: metaspace_images_dump\molformer_pubchem_embeddings.npy  shape=(30478, 768)
[DONE] index:      metaspace_images_dump\molformer_pubchem_index.parquet  rows=30,478


In [3]:
import pandas as pd
df = pd.read_parquet(r"metaspace_images_dump\molformer_pubchem_index.parquet")
df

,name,cid,smiles,inchi
0,(+)-(S)-Carvone,CID16724,CC1=CC[C@@H](CC1=O)C(=C)C,InChI=1S/C10H14O/c1-7(2)9-5-4-8(3)10(11)6-9/h4...
1,"(+)-1(10),4-Cadinadiene",CID441005,CC1=C[C@H]2[C@@H](CCC(=C2CC1)C)C(C)C,InChI=1S/C15H24/c1-10(2)13-8-6-12(4)14-7-5-11(...
2,"(+)-1(9),10-Pacifigorgiadiene",CID131752220,CC1CCC2C(CC=C2C1C=C(C)C)C,InChI=1S/C15H24/c1-10(2)9-15-12(4)5-7-13-11(3)...
3,"(+)-1,18-Nonacosanediol",CID86172616,CCCCCCCCCCCC(CCCCCCCCCCCCCCCCCO)O,InChI=1S/C29H60O2/c1-2-3-4-5-6-14-17-20-23-26-...
4,(+)-1-Methylpropyl 3-(methylthio)-2-propenyl d...,CID88158913,CCC(C)SSC/C=C/SC,InChI=1S/C8H16S3/c1-4-8(2)11-10-7-5-6-9-3/h5-6...
...,...,...,...,...
30473,{[4-(7-methoxy-2-oxo-2H-chromen-6-yl)butan-2-y...,CID131835273,CC(CCC1=C(C=C2C(=C1)C=CC(=O)O2)OC)OS(=O)(=O)O,"InChI=1S/C14H16O7S/c1-9(21-22(16,17)18)3-4-10-..."
30474,{[5-(4-methoxyphenyl)-2-methyl-3-oxopent-4-en-...,CID131836946,CC(COS(=O)(=O)O)C(=O)/C=C/C1=CC=C(C=C1)OC,"InChI=1S/C13H16O6S/c1-10(9-19-20(15,16)17)13(1..."
30475,{[5-(4-methoxyphenyl)-3-oxopentan-2-yl]oxy}sul...,CID131836979,CC(C(=O)CCC1=CC=C(C=C1)OC)OS(=O)(=O)O,"InChI=1S/C12H16O6S/c1-9(18-19(14,15)16)12(13)8..."
30476,{[5-(4-methoxyphenyl)-3-oxopentyl]oxy}sulfonic...,CID131836983,COC1=CC=C(C=C1)CCC(=O)CCOS(=O)(=O)O,InChI=1S/C12H16O6S/c1-17-12-6-3-10(4-7-12)2-5-...


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

MANIFEST = Path(r"metaspace_images_dump\manifest_REBUILT2_expanded_with_candidates_structures.parquet")
INDEX    = Path(r"metaspace_images_dump\molformer_pubchem_index.parquet")

OUT_FLAT    = Path(r"metaspace_images_dump\manifest_cand_molformer_rows_flat.npy")
OUT_OFFSETS = Path(r"metaspace_images_dump\manifest_cand_molformer_offsets.npy")
OUT_MANIFEST= Path(r"metaspace_images_dump\manifest_REBUILT2_with_molformer_candidates.parquet")

print("[LOAD] manifest...")
df = pd.read_parquet(MANIFEST)

print("[LOAD] molformer index...")
idx = pd.read_parquet(INDEX)

# ------------------------------------------------------------
# Robust CID parsing (handles: 16724, "16724", "CID16724", "cid16724", "CID 16724")
# ------------------------------------------------------------
def parse_cid_value(x):
    if x is None:
        return None
    # numeric types
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return None
        return int(x)

    s = str(x).strip()
    if not s:
        return None
    s_upper = s.upper()

    # strip "CID" prefix if present
    if s_upper.startswith("CID"):
        s = s[3:].strip()

    # sometimes "CID:16724" or "CID=16724"
    if s.startswith(":") or s.startswith("="):
        s = s[1:].strip()

    # keep only leading integer portion
    digits = []
    for ch in s:
        if ch.isdigit():
            digits.append(ch)
        else:
            break
    if not digits:
        return None
    try:
        return int("".join(digits))
    except Exception:
        return None

print("[BUILD] CID → molformer row map...")

# Your index may have "cid" as int or as "CID####"
# Use whichever column exists.
cid_col = None
for c in ["cid", "cid_used", "cid_str", "pubchem_cid"]:
    if c in idx.columns:
        cid_col = c
        break
if cid_col is None:
    raise KeyError(f"Could not find a CID column in INDEX. Columns: {list(idx.columns)}")

cid_to_row = {}
bad = 0
for i, v in enumerate(idx[cid_col].values):
    cid = parse_cid_value(v)
    if cid is None:
        bad += 1
        continue
    # first occurrence wins (stable)
    if cid not in cid_to_row:
        cid_to_row[cid] = i

print(f"[INFO] index rows={len(idx):,}  mapped_cids={len(cid_to_row):,}  bad_cids={bad:,}")

# ------------------------------------------------------------
# Parse manifest cand_pubchem_cids cells:
#   expected format now: "CID123 ; CID456 ; CID789"
# ------------------------------------------------------------
def parse_cid_rows(cell):
    if cell is None:
        return []
    s = str(cell).strip()
    if not s:
        return []

    parts = [p.strip() for p in s.split(" ; ") if p.strip()]
    rows = []
    for p in parts:
        cid = parse_cid_value(p)
        if cid is None:
            continue
        r = cid_to_row.get(cid)
        if r is not None:
            rows.append(r)

    # de-dup while keeping order
    seen = set()
    out = []
    for r in rows:
        if r not in seen:
            seen.add(r)
            out.append(r)
    return out

# column name sanity
if "cand_pubchem_cids" not in df.columns:
    raise KeyError(f"manifest missing 'cand_pubchem_cids'. Columns: {list(df.columns)}")

print("[PARSE] parsing cand_pubchem_cids → molformer row lists...")
cand_rows = [
    parse_cid_rows(cell)
    for cell in tqdm(df["cand_pubchem_cids"].values, total=len(df))
]

print("[BUILD] building flat + offsets arrays...")
flat = np.array([r for lst in cand_rows for r in lst], dtype=np.int32)

offsets = np.zeros(len(cand_rows) + 1, dtype=np.int64)
k = 0
for i, lst in enumerate(tqdm(cand_rows, desc="Offsets")):
    k += len(lst)
    offsets[i + 1] = k

print("[SAVE] writing arrays...")
OUT_FLAT.parent.mkdir(parents=True, exist_ok=True)
np.save(OUT_FLAT, flat)
np.save(OUT_OFFSETS, offsets)

print("[SAVE] writing manifest with counts...")
df2 = df.copy()
df2["n_cand_molformer"] = [len(lst) for lst in cand_rows]
df2.to_parquet(OUT_MANIFEST, index=False)

print("[DONE]")
print(f"  flat:    {OUT_FLAT}    shape={flat.shape} dtype={flat.dtype}")
print(f"  offsets: {OUT_OFFSETS} shape={offsets.shape} dtype={offsets.dtype}")
print(f"  manifest:{OUT_MANIFEST} rows={len(df2):,}")


[LOAD] manifest...
[LOAD] molformer index...
[BUILD] CID → molformer row map...
[INFO] index rows=30,478  mapped_cids=30,478  bad_cids=0
[PARSE] parsing cand_pubchem_cids → molformer row lists...


100%|██████████| 695500/695500 [00:19<00:00, 35426.32it/s]


[BUILD] building flat + offsets arrays...


Offsets: 100%|██████████| 695500/695500 [00:00<00:00, 2087114.05it/s]


[SAVE] writing arrays...
[SAVE] writing manifest with counts...
[DONE]
  flat:    metaspace_images_dump\manifest_cand_molformer_rows_flat.npy    shape=(7475864,) dtype=int32
  offsets: metaspace_images_dump\manifest_cand_molformer_offsets.npy shape=(695501,) dtype=int64
  manifest:metaspace_images_dump\manifest_REBUILT2_with_molformer_candidates.parquet rows=695,500


In [2]:
import pandas as pd
df = pd.read_parquet("metaspace_images_dump/manifest_REBUILT2_with_molformer_candidates.parquet")
df

,dataset_id,name,organism,split,db,fdr,msm,sum_formula,adduct,isotope_index,...,tile_stride,Organism_Part,Condition,sum_formula_clean,adduct_clean,cand_names,cand_smiles,cand_inchi,cand_pubchem_cids,n_cand_molformer
0,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957401,C10H14N5O8P,-H,0.0,...,NaN,Colon,"Cancer, xenograft",C10H14N5O8P,-H,Guanosine monophosphate; 8-Oxo-dGMP; Cyclic py...,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,InChI=1S/C10H14N5O8P/c11-10-13-7-4(8(18)14-10)...,CID135398631 ; CID135488904 ; CID135463437,3
1,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957401,C10H14N5O8P,-H,1.0,...,NaN,Colon,"Cancer, xenograft",C10H14N5O8P,-H,Guanosine monophosphate; 8-Oxo-dGMP; Cyclic py...,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,InChI=1S/C10H14N5O8P/c11-10-13-7-4(8(18)14-10)...,CID135398631 ; CID135488904 ; CID135463437,3
2,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957401,C10H14N5O8P,-H,2.0,...,NaN,Colon,"Cancer, xenograft",C10H14N5O8P,-H,Guanosine monophosphate; 8-Oxo-dGMP; Cyclic py...,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,InChI=1S/C10H14N5O8P/c11-10-13-7-4(8(18)14-10)...,CID135398631 ; CID135488904 ; CID135463437,3
3,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957267,C10H20O2,-H,0.0,...,NaN,Colon,"Cancer, xenograft",C10H20O2,-H,"(1R,2S,3S,4R)-p-Menthane-2,3-diol; p-Menthane-...",CC1CCC(C(C1O)O)C(C)C ; CC1CCC(C(C1)O)C(C)(C)O ...,InChI=1S/C10H20O2/c1-6(2)8-5-4-7(3)9(11)10(8)1...,CID107175 ; CID556998 ; CID5463962 ; CID12294 ...,32
4,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957267,C10H20O2,-H,1.0,...,NaN,Colon,"Cancer, xenograft",C10H20O2,-H,"(1R,2S,3S,4R)-p-Menthane-2,3-diol; p-Menthane-...",CC1CCC(C(C1O)O)C(C)C ; CC1CCC(C(C1)O)C(C)(C)O ...,InChI=1S/C10H20O2/c1-6(2)8-5-4-7(3)9(11)10(8)1...,CID107175 ; CID556998 ; CID5463962 ; CID12294 ...,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695495,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.867454,C45H78O2,+Na,1.0,...,NaN,Kidney,Control,C45H78O2,+Na,CE(18:1(9Z)); CE(18:1(11Z)); Palmitoylstigmast...,CCCCCCCC/C=C\CCCCCCCC(=O)O[C@H]1CC[C@@]2([C@H]...,InChI=1S/C45H78O2/c1-7-8-9-10-11-12-13-14-15-1...,CID5283632 ; CID53477793,2
695496,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.867454,C45H78O2,+Na,2.0,...,NaN,Kidney,Control,C45H78O2,+Na,CE(18:1(9Z)); CE(18:1(11Z)); Palmitoylstigmast...,CCCCCCCC/C=C\CCCCCCCC(=O)O[C@H]1CC[C@@]2([C@H]...,InChI=1S/C45H78O2/c1-7-8-9-10-11-12-13-14-15-1...,CID5283632 ; CID53477793,2
695497,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.899866,C47H95N2O6P,+Na,0.0,...,NaN,Kidney,Control,C47H95N2O6P,+Na,SM(d18:0/24:1(15Z)),CCCCCCCCCCCCCCC[C@H]([C@H](COP(=O)([O-])OCC[N+...,InChI=1S/C47H95N2O6P/c1-6-8-10-12-14-16-18-20-...,CID44260133,1
695498,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.899866,C47H95N2O6P,+Na,1.0,...,NaN,Kidney,Control,C47H95N2O6P,+Na,SM(d18:0/24:1(15Z)),CCCCCCCCCCCCCCC[C@H]([C@H](COP(=O)([O-])OCC[N+...,InChI=1S/C47H95N2O6P/c1-6-8-10-12-14-16-18-20-...,CID44260133,1
